                ## Decision Tree Learning ##
Write a Python function that implements the decision tree learning algorithm for classification. The function 
should use recursive binary splitting based on entropy and information gain to build a decision tree. It should take 
a list of examples (each example is a dict of attribute-value pairs) and a list of attribute names as input, 
and return a nested dictionary representing the decision tree.

In [1]:
import math
from collections import Counter

def entropy(examples: list[dict], target_attr: str) -> float:
    labels = [example[target_attr] for example in examples]
    label_counts = Counter(labels)
    total = len(labels)

    return -sum((count / total) * math.log2(count / total) for count in label_counts.values())


def information_gain(examples: list[dict], attribute: str, target_attr: str) -> float:
    total_entropy = entropy(examples, target_attr)
    total = len(examples)

    # Group by attribute values
    subsets = {}
    for example in examples:
        key = example[attribute]
        subsets.setdefault(key, []).append(example)

    weighted_entropy = sum(
        (len(subset) / total) * entropy(subset, target_attr)
        for subset in subsets.values()
    )

    return total_entropy - weighted_entropy


def majority_class(examples: list[dict], target_attr: str) -> str:
    labels = [example[target_attr] for example in examples]
    return Counter(labels).most_common(1)[0][0]


def all_same_class(examples: list[dict], target_attr: str) -> bool:
    labels = [example[target_attr] for example in examples]
    return all(label == labels[0] for label in labels)


def learn_decision_tree(examples: list[dict], attributes: list[str], target_attr: str) -> dict:
    # Base case 0
    if not examples:
        return None
        
    # Base case 1: if no attrubute to split
    if len(attributes) == 0:
        return majority_class(examples, target_attr)

    # Base case 2 : all examples have same label
    if all_same_class(examples, target_attr):
        return examples[0][target_attr]

    # Get the best attribute to split
    #best_attr = max(attributes, key=lambda attr: information_gain(examples, attr, target_attr))
    best_attr = max(attributes, key=lambda attr: information_gain(examples, attr, target_attr))
                    
    # Make a tree node for best_attr
    tree = {best_attr: {}}

    # Get the distinct values for best_attr
    attr_values = set([example[best_attr] for example in examples])
    #print(attr_values)
    for value in attr_values:
        # Find samples with sample[best_attr] == value
        subset = [ex for ex in examples if ex[best_attr] == value]
        
        # Compute the remaining attributes
        remaining_attrs = [attr for attr in attributes if attr != best_attr]

        # Recursive call
        subtree = learn_decision_tree( subset, remaining_attrs, target_attr)

        # Assign subtree to corresponding value
        tree[best_attr][value] =subtree
    return tree

In [2]:
examples = [
                    {'Outlook': 'Sunny', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Weak', 'PlayTennis': 'No'},
                    {'Outlook': 'Sunny', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Strong', 'PlayTennis': 'No'},
                    {'Outlook': 'Overcast', 'Temperature': 'Hot', 'Humidity': 'High', 'Wind': 'Weak', 'PlayTennis': 'Yes'},
                    {'Outlook': 'Rain', 'Temperature': 'Mild', 'Humidity': 'High', 'Wind': 'Weak', 'PlayTennis': 'Yes'}
                ]


In [3]:
attributes = ['Outlook', 'Temperature', 'Humidity', 'Wind']

In [5]:
learn_decision_tree(examples, attributes, target_attr = 'PlayTennis')

{'Outlook': {'Overcast': 'Yes', 'Sunny': 'No', 'Rain': 'Yes'}}